In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [2]:
def initializeDriver():
    driver = webdriver.Chrome()
    driver.get('https://it.whoscored.com/')
    acceptCookie(driver)
    return driver

def acceptCookie(driver):
    try:
        time.sleep(1)
        cookie_button = driver.find_element(By.XPATH, '//*[@id="qc-cmp2-ui"]/div[2]/div/button[2]')
        cookie_button.click()
        time.sleep(1)
        button = driver.find_element(By.CLASS_NAME, 'webpush-swal2-close')
        button.click()
    except:
        pass

        
def getPlayerStats(driver,team):
    statistics_table = driver.find_element(By.XPATH, '//*[@id="live-player-stats"]')
    stats = statistics_table.find_element(By.ID, f"live-player-{team}-stats")
    stats_div = stats.find_elements(By.CLASS_NAME, 'statistics-table-tab')
    table_buttons = getButtons(driver,team)
    player_stats = {}
    for div in stats_div:
        type_div = div.get_property('id').split('-')[-1]
        table_buttons[type_div].click()
        time.sleep(1)   
        tab = div.find_element(By.TAG_NAME, 'table').find_elements(By.TAG_NAME,'tbody')[0]
        records = tab.find_elements(By.TAG_NAME, 'tr')
        for i in range(len(records)):
            player_td = records[i].find_elements(By.CLASS_NAME, 'player-link')
            player = player_td[0].find_element(By.CLASS_NAME, 'iconize').text
            cols = records[i].find_elements(By.TAG_NAME,'td')
            d={}
            for c in cols:
                d[ c.get_attribute('class')] = c.text
                #print(player, c.get_attribute('class'), c.text)
            if player not in player_stats.keys():
                player_stats[player] = d
            else:
                player_stats[player] = player_stats[player] | d
    return player_stats

def getButtons(driver, team):
    traduzioni = {'Riepilogo': 'summary', 'Attacco': 'offensive', 'Difesa': 'defensive', 'Passaggi': 'passing'}
    statistics_table = driver.find_element(By.XPATH, '//*[@id="live-player-stats"]')
    table_buttons = {}
    for i in range(1,5):
        time.sleep(1)
        button = statistics_table.find_element(By.XPATH, f'//*[@id="live-player-{team}-options"]/li[{i}]/a')
        button_name = button.text
        table_buttons[traduzioni[button_name]] = button

    return table_buttons

def getTeams(driver):
    home = driver.find_element(By.XPATH, '//*[@id="match-header"]/div/div[1]/span[1]').text
    away = driver.find_element(By.XPATH, '//*[@id="match-header"]/div/div[1]/span[5]').text
    return home, away

def elaboraStatsMatch(driver, url, out_dir):
    driver.get(url)
    time.sleep(1)
    acceptCookie(driver)
    stats_button = driver.find_element(By.XPATH, '//*[@id="sub-sub-navigation"]/ul/li[2]/a')
    stats_button.click()
    time.sleep(1)
    match_stats = {}
    home, away = getTeams(driver)
    match_stats[home] = getPlayerStats(driver, 'home')
    match_stats[away] = getPlayerStats(driver, 'away')
    writeJson(match_stats, os.path.join(out_dir, home + '_' + away + '.json'))

def getFirstMatchDay(driver):
    match_button = driver.find_element(By.XPATH, '//*[@id="sub-navigation"]/ul/li[2]/a')
    match_button.click()
    calendar = driver.find_element(By.XPATH, '//*[@id="toggleCalendar"]')
    calendar.click()
    calendar_year = driver.find_element(By.XPATH, '//*[@id="datePicker"]/div/div[1]/div/div/button')
    calendar_year.click()
    try:
        first_year = driver.find_element(By.XPATH, '//*[@id="datePicker"]/div/div[1]/table/tbody/tr/td[2]')
        first_year.click()
    except:
        first_year = driver.find_element(By.XPATH, '//*[@id="datePicker"]/div/div[1]/table/tbody/tr/td[1]')
        first_year.click()

    for i in range(1,5):
        for j in range(1,4):
            try:
                first_month = driver.find_element(By.XPATH, f'//*[@id="datePicker"]/div/div[1]/table/tbody/tr[{i}]/td[{j}]')
                first_month.click()
                break
            except:
                continue


def getLeagueMatches(driver):
    match_list = []
    getFirstMatchDay(driver)
    time.sleep(1)
    current_month = driver.find_element(By.XPATH, '//*[@id="toggleCalendar"]/span[1]').text
    next_month = ''
    while current_month != next_month:
        matches = driver.find_elements(By.CLASS_NAME, 'Match-module_row__zwBOn')
        match_list_current = [match.find_element(By.TAG_NAME, 'a').get_attribute('href') for match in matches]
        match_list = match_list + match_list_current
        current_month = driver.find_element(By.XPATH, '//*[@id="toggleCalendar"]/span[1]').text
        next_month_button = driver.find_element(By.XPATH, '//*[@id="dayChangeBtn-next"]')
        next_month_button.click()
        next_month = driver.find_element(By.XPATH, '//*[@id="toggleCalendar"]/span[1]').text
        time.sleep(1)
    return match_list

    
def isMatchComputed(league, season, match):
    dir = os.path.join('Dataset','WhoScored')
    dir = os.path.join(dir ,league)
    dir = os.path.join(dir ,season)
    match = match.replace('AC-Milan','AC Milan').replace('-','_')
    file = os.path.join(dir ,f'{match}.json')
    return os.path.isfile(file)

In [3]:
def getLeagueUrls(minimize_window=True):
    
    main_url = 'https://it.whoscored.com/'
    driver = initializeDriver()

    if minimize_window:
        driver.minimize_window()

    league_names = []
    league_urls = []
    
    tournaments_btn = driver.find_element(By.XPATH, '//*[@id="Tutti-i-Tornei-btn"]').click()
    n_button = soup(driver.find_element(By.XPATH, '//*[@id="header-wrapper"]/div/div/div/div[4]/div[2]/div/div/div/div[1]/div/div').get_attribute('innerHTML')).find_all('button')
    n_tournaments = []
    for button in n_button:
        id_button = button.get('id')
        driver.find_element(By.ID, id_button).click()
        n_country = soup(driver.find_element(By.XPATH, '//*[@id="header-wrapper"]/div/div/div/div[4]/div[2]/div/div/div/div[2]').get_attribute('innerHTML')).find_all('div', {'class':'TournamentsDropdownMenu-module_countryDropdownContainer__I9P6n'})

        for country in n_country:
            country_id = country.find('div', {'class': 'TournamentsDropdownMenu-module_countryDropdown__8rtD-'}).get('id')

            # Trouver l'élément avec Selenium et cliquer dessus
            country_element = driver.find_element(By.ID, country_id)
            country_element.click()

            html_tournaments_list = driver.find_element(By.XPATH, '//*[@id="header-wrapper"]/div/div/div/div[4]/div[2]/div/div/div/div[2]').get_attribute('innerHTML')

            # Parse le HTML avec BeautifulSoup pour trouver les liens des tournois
            soup_tournaments = soup(html_tournaments_list, 'html.parser')
            tournaments = soup_tournaments.find_all('a')

            # Ajouter les tournois à la liste n_tournaments
            n_tournaments.extend(tournaments)

            driver.execute_script("arguments[0].click();", country_element)


    for tournament in n_tournaments:
        league_name = tournament.get('href').split('/')[-1]
        league_link = main_url[:-1]+tournament.get('href')
        league_names.append(league_name)
        league_urls.append(league_link)

    leagues = {}
    for name,link in zip(league_names,league_urls):
        leagues[name] = link

    driver.close()
    return leagues

def getMatchUrls(comp_urls, competition, season, maximize_window=True):

    driver = initializeDriver()
    
    if maximize_window:
        driver.maximize_window()
    
    comp_url = comp_urls[competition]
    driver.get(comp_url)
    time.sleep(5)
    
    seasons = driver.find_element(By.XPATH, '//*[@id="seasons"]').get_attribute('innerHTML').split(sep='\n')
    seasons = [i for i in seasons if i]
    
    
    for i in range(1, len(seasons)+1):
        if driver.find_element(By.XPATH, '//*[@id="seasons"]/option['+str(i)+']').text == season:
            driver.find_element(By.XPATH, '//*[@id="seasons"]/option['+str(i)+']').click()
            
            time.sleep(5)
            try:
                stages = driver.find_element(By.XPATH, '//*[@id="stages"]').get_attribute('innerHTML').split(sep='\n')
                stages = [i for i in stages if i]
                
                all_urls = []
            
                for i in range(1, len(stages)+1):
                    print(driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').text)
                    if competition == 'Champions League' or competition == 'Europa League':
                        if 'Grp' in driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').text or 'Final Stage' in driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').text:
                            driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').click()
                            time.sleep(5)
                            
                            driver.execute_script("window.scrollTo(0, 400)") 
                            
                            match_urls = getFixtureData(driver)
                            
                            match_urls = getSortedData(match_urls)
                            
                            match_urls2 = [url for url in match_urls if '?' not in url['date'] and '\n' not in url['date']]
                            
                            all_urls += match_urls2
                        else:
                            continue
                    
                    elif competition == 'Major League Soccer':
                        if 'Grp. ' not in driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').text: 
                            driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').click()
                            time.sleep(5)
                        
                            driver.execute_script("window.scrollTo(0, 400)")
                            
                            match_urls = getFixtureData(driver)
                            
                            match_urls = getSortedData(match_urls)
                            
                            match_urls2 = [url for url in match_urls if '?' not in url['date'] and '\n' not in url['date']]
                            
                            all_urls += match_urls2
                        else:
                            continue
                        
                    else:
                        driver.find_element(By.XPATH, '//*[@id="stages"]/option['+str(i)+']').click()
                        time.sleep(5)
                    
                        driver.execute_script("window.scrollTo(0, 400)")
                        
                        match_urls = getFixtureData(driver)
                        
                        match_urls = getSortedData(match_urls)
                        
                        match_urls2 = [url for url in match_urls if '?' not in url['date'] and '\n' not in url['date']]
                        
                        all_urls += match_urls2
                
            except NoSuchElementException:
                all_urls = []
                
                driver.execute_script("window.scrollTo(0, 400)")
                
                match_urls = getFixtureData(driver)
                
                match_urls = getSortedData(match_urls)
                
                match_urls2 = [url for url in match_urls if '?' not in url['date'] and '\n' not in url['date']]
                
                all_urls += match_urls2
            
            
            remove_dup = [dict(t) for t in {tuple(sorted(d.items())) for d in all_urls}]
            all_urls = getSortedData(remove_dup)
            
            driver.close() 
    
            return all_urls
     
    season_names = [re.search(r'\>(.*?)\<',season).group(1) for season in seasons]
    driver.close() 
    print('Seasons available: {}'.format(season_names))
    raise('Season Not Found.')

def getFixtureData(driver):
    matches_ls = []
    while True:
        initial = driver.page_source
        all_fixtures = driver.find_elements(By.CLASS_NAME, 'Accordion-module_accordion__UuHD0')
        for dates in all_fixtures:
            fixtures = dates.find_elements(By.CLASS_NAME, 'Match-module_row__zwBOn')
            date_row = dates.find_element(By.CLASS_NAME, 'Accordion-module_header__HqzWD')
            for row in fixtures:
                url = row.find_element(By.TAG_NAME, 'a')
                if 'Live' in url.get_attribute('href'):
                    match_dict = {}
                    element = soup(row.get_attribute('innerHTML'), features='lxml')
                    teams_tag = element.find("div", {"class":"Match-module_teams__sGVeq"})
                    link_tag = element.find("a")
                    match_dict['date'] = date_row.text
                    match_dict['home'] = teams_tag.find_all('a')[0].text
                    match_dict['away'] = teams_tag.find_all('a')[1].text
                    match_dict['score'] = ':'.join([t.text for t in link_tag.find_all('span')])
                    match_dict['url'] = link_tag['href']
                    matches_ls.append(match_dict)
        prev_btn = driver.find_element(By.ID, 'dayChangeBtn-prev')
        prev_btn.click()
        time.sleep(1)
        final = driver.page_source
        if initial == final:
            break

    return matches_ls

def getSortedData(data):
    data = sorted(data, key = lambda i: dt.strptime(i['date'], '%A, %b %d %Y'))
    return data


In [17]:
try:
    dir = os.path.join('Dataset', 'WhoScored')
    selected_seasons = ['2023/2024', '2022/2023','2021/2022','2020/2021','2019/2020']
    league = 'Serie A'
    url = 'https://it.whoscored.com/Regions/108/Tournaments/5/Italia-Serie-A'
    out_dir = os.path.join(dir,league)
    os.makedirs(out_dir, exist_ok=True)

    driver = webdriver.Chrome()
    driver.get(url)
    acceptCookie(driver)
    seasons = list(driver.find_element(By.XPATH, '//*[@id="seasons"]').find_elements(By.TAG_NAME, 'option'))
    season_urls = {}
    for s in seasons:
        if s.text in selected_seasons:
            s_key = s.text.replace('/','-')
            os.makedirs(os.path.join(out_dir, s_key), exist_ok=True)
            season_urls[s_key] = 'https://it.whoscored.com/'+s.get_attribute('value')

    for s_key, s in season_urls.items():
        print(s_key, s)
        driver.get(s)
        stages = driver.find_elements(By.XPATH, '//*[@id="stages"]')
        if len(stages) == 1:
            time.sleep(1)
            opt = driver.find_element(By.XPATH, '//*[@id="stages"]/option[1]')
            opt.click()
        
        out_dir_s = os.path.join(out_dir, s_key)
        getFirstMatchDay(driver)
        match_list = getLeagueMatches(driver)
        print(len(match_list))
        for match in match_list:
            elaboraStatsMatch(driver, match, out_dir_s)
        time.sleep(1)
        
    driver.quit()
except Exception as e :
    print(e)        
    driver.quit()

2023-2024 https://it.whoscored.com//Regions/108/Tournaments/5/Seasons/9659/Italia-Serie-A
380
Message: element click intercepted: Element <a href="/Matches/1746274/LiveStatistics/Italia-Serie-A-2023-2024-Salernitana-Empoli" class="">...</a> is not clickable at point (596, 391). Other element would receive the click: <div class="webpush-swal2-container webpush-swal2-center webpush-swal2-fade webpush-swal2-shown" style="background: rgba(0, 0, 0, 0.4); overflow-y: auto;">...</div>
  (Session info: chrome=130.0.6723.70)
Stacktrace:
	GetHandleVerifier [0x00007FF7D6973AB5+28005]
	(No symbol) [0x00007FF7D68D83B0]
	(No symbol) [0x00007FF7D677580A]
	(No symbol) [0x00007FF7D67CD6CE]
	(No symbol) [0x00007FF7D67CB16C]
	(No symbol) [0x00007FF7D67C8628]
	(No symbol) [0x00007FF7D67C785D]
	(No symbol) [0x00007FF7D67B990E]
	(No symbol) [0x00007FF7D67EBA3A]
	(No symbol) [0x00007FF7D67B9246]
	(No symbol) [0x00007FF7D67EBC50]
	(No symbol) [0x00007FF7D680B8B3]
	(No symbol) [0x00007FF7D67EB7E3]
	(No symbol)

In [3]:
def writeMatchesList(selected_seasons, league, competitions):

    dir = os.path.join('Dataset', 'WhoScored')

    #url = 'https://it.whoscored.com/Regions/108/Tournaments/5/Italia-Serie-A'
    out_dir = os.path.join(dir,league)
    os.makedirs(out_dir, exist_ok=True)
    url = competitions[league]
    driver = initializeDriver()
    driver.get(url)
    seasons = list(driver.find_element(By.XPATH, '//*[@id="seasons"]').find_elements(By.TAG_NAME, 'option'))
    season_urls = {}
    for s in seasons:
        if s.text in selected_seasons:
            s_key = s.text.replace('/','-')
            os.makedirs(os.path.join(out_dir, s_key), exist_ok=True)
            season_urls[s_key] = 'https://it.whoscored.com/'+s.get_attribute('value')

    for s_key, s in season_urls.items():
        print(s_key, s)
        driver.get(s)
        stages = driver.find_elements(By.XPATH, '//*[@id="stages"]')
        if len(stages) == 1:
            select = Select(stages[0])
            time.sleep(1)
            match_list = []
            urls = ['http://it.whoscored.com'+option.get_attribute('value') for option in select.options]
            for url in urls:
                driver.get(url)
                getFirstMatchDay(driver)
                match_list = match_list + getLeagueMatches(driver)
        else:
            getFirstMatchDay(driver)
            match_list =  getLeagueMatches(driver)

        
        out_dir_s = os.path.join(out_dir, s_key)
        writeJson(match_list, os.path.join(dir, f'{league}_{s_key}.json'))
            


In [30]:
leagues['Paesi-Bassi-Eredivisie']

dict_keys(['Africa-CAF-Champions-League', 'Africa-CAF-Super-Cup', 'Africa-Africa-Cup-of-Nations-Qualification', 'Africa-African-Nations-Championship-Qualification', 'Africa-CAF-Champions-League-Qualification', 'Africa-CAF-Confederations-Cup', 'Africa-CECAFA-Senior-Challenge-Cup', 'Asia-Gulf-Cup', 'Asia-East-Asian-Championship', 'Asia-AFF-Championship', 'Asia-AFC-Champions-League', 'Asia-AFC-Cup', 'Asia-AFC-Champions-League-Elite-Qualification', 'Asia-AFC-U23-Asian-Cup', 'Asia-ASEAN-Championship-Qualification', 'Asia-Asian-Cup-Qualification', 'Asia-Premier-League-Asia-Trophy', 'Europa-Champions-League', 'Europa-Europa-League', 'Europa-European-Championship-Qualification', 'Europa-UEFA-Super-Cup', 'Europa-Conference-League', 'Europa-Women-s-Champions-League', 'Europa-Champions-League-Qualification', 'Europa-Europa-League-Qualification', 'Europa-Conference-League-Qualification', 'Europa-UEFA-Youth-League', 'Europa-The-Atlantic-Cup', 'Internazionale-European-Championship', 'Internazionale-

In [32]:
selected_seasons = ['2023/2024', '2022/2023','2021/2022','2020/2021','2019/2020']
#selected_seasons = ['2022/2023']
selected_leagues =['Europa-Champions-League','Europa-Europa-League', 'Francia-Ligue-1', 'Germania-Bundesliga', 'Spagna-LaLiga','Inghilterra-Premier-League', 'Portogallo-Liga-Portugal', 'Paesi-Bassi-Eredivisie']
#leagues = getLeagueUrls()
for l in selected_leagues:
    writeMatchesList(selected_seasons, l, leagues)

2023-2024 https://it.whoscored.com//Regions/250/Tournaments/12/Seasons/9664/Europa-Champions-League
2022-2023 https://it.whoscored.com//Regions/250/Tournaments/12/Seasons/9086/Europa-Champions-League
2021-2022 https://it.whoscored.com//Regions/250/Tournaments/12/Seasons/8623/Europa-Champions-League
2020-2021 https://it.whoscored.com//Regions/250/Tournaments/12/Seasons/8177/Europa-Champions-League
2019-2020 https://it.whoscored.com//Regions/250/Tournaments/12/Seasons/7804/Europa-Champions-League
2023-2024 https://it.whoscored.com//Regions/250/Tournaments/30/Seasons/9778/Europa-Europa-League
2022-2023 https://it.whoscored.com//Regions/250/Tournaments/30/Seasons/9087/Europa-Europa-League
2021-2022 https://it.whoscored.com//Regions/250/Tournaments/30/Seasons/8741/Europa-Europa-League
2020-2021 https://it.whoscored.com//Regions/250/Tournaments/30/Seasons/8178/Europa-Europa-League
2019-2020 https://it.whoscored.com//Regions/250/Tournaments/30/Seasons/7805/Europa-Europa-League
2023-2024 https

: 

In [9]:
url = 'https://it.whoscored.com/Regions/108/Tournaments/5/Seasons/9159/Italia-Serie-A'
driver = initializeDriver()
driver.get(url)

In [18]:
driver.get(url)
stages = driver.find_elements(By.XPATH, '//*[@id="stages"]')
select = Select(stages[0])
options = select.options
for option in options:
    print(option.text, option.get_attribute('value'))
    print('http://it.whoscored.com'+option.get_attribute('value'))
    #driver.get('http://it.whoscored.com'+option.get_attribute('value'))
    #time.sleep(0.1)


Serie A /Regions/108/Tournaments/5/Seasons/9159/Stages/21087/Show/Italia-Serie-A-2022-2023
http://it.whoscored.com/Regions/108/Tournaments/5/Seasons/9159/Stages/21087/Show/Italia-Serie-A-2022-2023
Serie A Relegation Playoff /Regions/108/Tournaments/5/Seasons/9159/Stages/22100/Show/Italia-Serie-A-2022-2023
http://it.whoscored.com/Regions/108/Tournaments/5/Seasons/9159/Stages/22100/Show/Italia-Serie-A-2022-2023


In [4]:
#selected_seasons = ['2023-2024', '2022-2023','2021-2022','2020-2021','2019-2020']
selected_seasons = ['2022-2023','2021-2022','2020-2021','2019-2020']
league = 'Serie A'
#season = selected_seasons[0]
dir = os.path.join('Dataset', 'WhoScored')
out_dir = os.path.join(dir,league)
for season in selected_seasons:
    out_dir_s = os.path.join(out_dir, season)
    file = os.path.join(dir,f'{league}_{season}.json')
    matches = readJson(file)
    sep = league.replace(' ', '-') + '-' + season + '-'
    driver = initializeDriver()
    for url in matches:
        match = url.split(sep)[1]
        if not isMatchComputed(league, season, match):
            try:
                elaboraStatsMatch(driver, url, out_dir_s)
            except:
                print(f'Errore in {match}, stagione {season}')

driver.quit()

Errore in Napoli-Spezia, stagione 2022-2023
Errore in Empoli-Atalanta, stagione 2022-2023
Errore in Cremonese-Monza, stagione 2022-2023
Errore in Cremonese-Atalanta, stagione 2022-2023
Errore in Verona-Sassuolo, stagione 2022-2023
Errore in Atalanta-Spezia, stagione 2022-2023
Errore in Salernitana-Atalanta, stagione 2021-2022
Errore in Napoli-Bologna, stagione 2021-2022
Errore in Atalanta-Lazio, stagione 2021-2022
Errore in Sassuolo-Empoli, stagione 2021-2022
Errore in Cagliari-Atalanta, stagione 2021-2022
Errore in Napoli-Verona, stagione 2021-2022
Errore in Parma-Calcio-1913-Udinese, stagione 2019-2020


: 

In [5]:
def getEventsMatch(driver, url):
    driver.get(url)
    script_content = driver.find_element(By.XPATH, '//*[@id="layout-wrapper"]/script[1]').get_attribute('innerHTML')
    script_content = re.sub(r"[\n\t]*", "", script_content)
    script_content = script_content[script_content.index("matchId"):script_content.rindex("}")]
    script_content_list = list(filter(None, script_content.strip().split(',            ')))
    metadata = script_content_list.pop(1)
    event_metadata = script_content_list.pop(1)
    match_data = json.loads(metadata[metadata.index('{'):])
    event_metadata = json.loads(event_metadata[event_metadata.index('{'):])
    return match_data, event_metadata

def writeEvents(match_data, out_dir):
    players_dict = match_data['playerIdNameDictionary']
    home = match_data['home']['name'].replace('/','_')
    away =  match_data['away']['name'].replace('/','_')
    date = match_data['timeStamp'].split()[0]
    teams = {match_data['home']['teamId']: home, match_data['away']['teamId']: away}
    i = 0
    events = match_data['events']
    nome_file = f'{date}_{home}_{away}.json'
    out_file = os.path.join(out_dir, nome_file)
    for event in events:
        try:
            player = players_dict[str(event['playerId'])]
            event['teamName'] = teams[event['teamId']]
            event['playerName'] = player
        except:
            continue
    out_file = os.path.join(out_dir, nome_file)
    writeJson(events, out_file)

In [6]:
driver = initializeDriver()
match_data, event_metadata = getEventsMatch(driver, 'https://it.whoscored.com/Matches/1495457/Live/Italia-Serie-A-2020-2021-Inter-AC-Milan')
dir = os.path.join('Dataset', 'WhoScored')
writeJson(event_metadata, os.path.join(dir, 'event_metadata.json'))

In [9]:
dir = os.path.join('Dataset', 'WhoScored')
subdir = os.path.join('Italia-Serie-A', '2020-2021')
out_dir = os.path.join(dir, subdir)
writeEvents(match_data, out_dir)

In [6]:
dir = os.path.join('Dataset', 'WhoScored')
driver = initializeDriver()
url_non_lavorati = []
for file in tqdm(os.listdir(dir), desc="Processing JSON files"):
    if file.endswith('.json'):
        dir_list = file.split('_')
        subdir = os.path.join(dir_list[0], dir_list[1].split('.')[0])
        out_dir = os.path.join(dir, subdir)
        match_urls = readJson(os.path.join(dir, file))
        for url in tqdm(match_urls, desc=f"Processing URLs in {file}", leave=False):
            try:
                match_data, _ = getEventsMatch(driver, url)
                writeEvents(match_data, out_dir)
            except:
                print(subdir, url)
                url_non_lavorati.append(url)
driver.quit()
writeJson(url_non_lavorati, os.path.join(dir, 'urlNonLavorati.json'))

Processing JSON files:  11%|█         | 6/56 [53:57<7:55:05, 570.11s/it]

Europa-Europa-League\2019-2020 https://it.whoscored.com/Matches/1456775/Live/Europa-Europa-League-2019-2020-Getafe-Inter


Europa-Europa-League\2019-2020 https://it.whoscored.com/Matches/1456776/Live/Europa-Europa-League-2019-2020-Roma-Sevilla


Processing JSON files:  14%|█▍        | 8/56 [1:11:40<7:21:10, 551.48s/it]

Europa-Europa-League\2020-2021 https://it.whoscored.com/Matches/1509039/Live/Europa-Europa-League-2020-2021-Villarreal-Qarabag-FK


...

Processing JSON files:  82%|████████▏ | 46/56 [13:53:43<2:47:44, 1006.41s/it]

Portogallo-Liga-Portugal\2020-2021 https://it.whoscored.com/Matches/1544925/Show/Portogallo-Liga-Portugal-2020-2021-Rio-Ave-Arouca


Processing JSON files:  93%|█████████▎| 52/56 [15:55:37<1:17:32, 1163.01s/it]

Spagna-LaLiga\2021-2022 https://it.whoscored.com/Matches/1559829/Live/Spagna-LaLiga-2021-2022-Athletic-Club-Barcelona


Processing JSON files: 100%|██████████| 56/56 [17:24:34<00:00, 1119.19s/it]  


In [43]:
event_data2 = {v: k for k,v in event_data.items()}
for sat in events[100]['satisfiedEventsTypes']:
    print(event_data2[sat])

touches
passAccurate
shortPassAccurate
passForward
passRight
defensiveThird
pos


: 

In [10]:
dir = os.path.join('Dataset', 'WhoScored')
out_dir = os.path.join(dir, 'Tmp')
driver = initializeDriver()
url_non_lavorati = []

match_urls = readJson(os.path.join(dir, 'urlNonLavorati.json'))
for url in tqdm(match_urls, desc=f"Processing URLs in {file}", leave=False):
    try:
        match_data, _ = getEventsMatch(driver, url)
        writeEvents(match_data, out_dir)
    except:
        print(url)
        url_non_lavorati.append(url)
driver.quit()
writeJson(url_non_lavorati, os.path.join(dir, 'urlNonLavorati_v2.json'))

Processing URLs in Stats:   1%|          | 1/82 [00:45<1:00:48, 45.05s/it]

https://it.whoscored.com/Matches/1509039/Live/Europa-Europa-League-2020-2021-Villarreal-Qarabag-FK


Processing URLs in Stats:   2%|▏         | 2/82 [00:53<31:32, 23.65s/it]  

https://it.whoscored.com/Matches/1627583/Live/Europa-Europa-League-2021-2022-RB-Leipzig-Spartak-Moscow


Processing URLs in Stats:   4%|▎         | 3/82 [00:56<18:45, 14.24s/it]

...

Processing URLs in Stats:  98%|█████████▊| 80/82 [05:57<00:03,  1.60s/it]

https://it.whoscored.com/Matches/1544924/Show/Portogallo-Liga-Portugal-2020-2021-Arouca-Rio-Ave


Processing URLs in Stats:  99%|█████████▉| 81/82 [05:59<00:01,  1.67s/it]

https://it.whoscored.com/Matches/1544925/Show/Portogallo-Liga-Portugal-2020-2021-Rio-Ave-Arouca


https://it.whoscored.com/Matches/1559829/Live/Spagna-LaLiga-2021-2022-Athletic-Club-Barcelona


In [42]:
events = match_data['events']
events[100], match_data['playerIdNameDictionary'][str(events[100]['playerId'])]

({'id': 2744726015.0,
  'eventId': 71,
  'minute': 4,
  'second': 55,
  'teamId': 52,
  'playerId': 379868,
  'x': 30.6,
  'y': 54.1,
  'expandedMinute': 4,
  'period': {'value': 1, 'displayName': 'FirstHalf'},
  'type': {'value': 1, 'displayName': 'Pass'},
  'outcomeType': {'value': 1, 'displayName': 'Successful'},
  'qualifiers': [{'type': {'value': 212, 'displayName': 'Length'},
    'value': '18.6'},
   {'type': {'value': 56, 'displayName': 'Zone'}, 'value': 'Back'},
   {'type': {'value': 178, 'displayName': 'StandingSave'}},
   {'type': {'value': 141, 'displayName': 'PassEndY'}, 'value': '26.7'},
   {'type': {'value': 140, 'displayName': 'PassEndX'}, 'value': '31.1'},
   {'type': {'value': 213, 'displayName': 'Angle'}, 'value': '4.74'}],
  'satisfiedEventsTypes': [91, 117, 30, 36, 38, 215, 218],
  'isTouch': True,
  'endX': 31.1,
  'endY': 26.7},
 'Jude Bellingham')

In [35]:
events = driver.find_elements(By.XPATH, '//*[@id="player-event-details"]')

In [23]:
import warnings
import time
import pandas as pd
pd.options.mode.chained_assignment = None
import json
from bs4 import BeautifulSoup as soup
import re 
from collections import OrderedDict
import datetime
from datetime import datetime as dt
import itertools
import numpy as np
try:
    from tqdm import trange
except ModuleNotFoundError:
    pass
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

options = webdriver.ChromeOptions()

options.add_experimental_option('excludeSwitches', ['enable-logging'])

def getMatchData(driver, url, display=True, close_window=True):
    try:
        driver.get(url)
    except WebDriverException:
        driver.get(url)

    time.sleep(5)
    # get script data from page source
    script_content = driver.find_element(By.XPATH, '//*[@id="layout-wrapper"]/script[1]').get_attribute('innerHTML')


    # clean script content
    script_content = re.sub(r"[\n\t]*", "", script_content)
    script_content = script_content[script_content.index("matchId"):script_content.rindex("}")]


    # this will give script content in list form 
    script_content_list = list(filter(None, script_content.strip().split(',            ')))
    metadata = script_content_list.pop(1) 


    # string format to json format
    match_data = json.loads(metadata[metadata.index('{'):])
    keys = [item[:item.index(':')].strip() for item in script_content_list]
    values = [item[item.index(':')+1:].strip() for item in script_content_list]
    for key,val in zip(keys, values):
        match_data[key] = json.loads(val)


    # get other details about the match
    region = driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/span[1]').text
    league = driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/a').text.split(' - ')[0]
    season = driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/a').text.split(' - ')[1]
    if len(driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/a').text.split(' - ')) == 2:
        competition_type = 'League'
        competition_stage = ''
    elif len(driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/a').text.split(' - '))== 3:
        competition_type = 'Knock Out'
        competition_stage = driver.find_element(By.XPATH, '//*[@id="breadcrumb-nav"]/a').text.split(' - ')[-1]
    else:
        print('Getting more than 3 types of information about the competition.')

    match_data['region'] = region
    match_data['league'] = league
    match_data['season'] = season
    match_data['competitionType'] = competition_type
    match_data['competitionStage'] = competition_stage


    # sort match_data dictionary alphabetically
    match_data = OrderedDict(sorted(match_data.items()))
    match_data = dict(match_data)
    if display:
        print('Region: {}, League: {}, Season: {}, Match Id: {}'.format(region, league, season, match_data['matchId']))
    
    
    if close_window:
        driver.close()
        
    return match_data

def createEventsDF(data):
    events = data['events']
    for event in events:
        event.update({'matchId' : data['matchId'],
                        'startDate' : data['startDate'],
                        'startTime' : data['startTime'],
                        'score' : data['score'],
                        'ftScore' : data['ftScore'],
                        'htScore' : data['htScore'],
                        'etScore' : data['etScore'],
                        'venueName' : data['venueName'],
                        'maxMinute' : data['maxMinute']})
    events_df = pd.DataFrame(events)

    # clean period column
    events_df['period'] = pd.json_normalize(events_df['period'])['displayName']

    # clean type column
    events_df['type'] = pd.json_normalize(events_df['type'])['displayName']

    # clean outcomeType column
    events_df['outcomeType'] = pd.json_normalize(events_df['outcomeType'])['displayName']

    # clean outcomeType column
    try:
        x = events_df['cardType'].fillna({i: {} for i in events_df.index})
        events_df['cardType'] = pd.json_normalize(x)['displayName'].fillna(False)
    except KeyError:
        events_df['cardType'] = False

    eventTypeDict = data['matchCentreEventTypeJson']  
    events_df['satisfiedEventsTypes'] = events_df['satisfiedEventsTypes'].apply(lambda x: [list(eventTypeDict.keys())[list(eventTypeDict.values()).index(event)] for event in x])

    # clean qualifiers column
    try:
        for i in events_df.index:
            row = events_df.loc[i, 'qualifiers'].copy()
            if len(row) != 0:
                for irow in range(len(row)):
                    row[irow]['type'] = row[irow]['type']['displayName']
    except TypeError:
        pass


    # clean isShot column
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        if 'isShot' in events_df.columns:
            events_df['isShot'] = events_df['isShot'].replace(np.nan, False).infer_objects()
        else:
            events_df['isShot'] = False

        # clean isGoal column
        if 'isGoal' in events_df.columns:
            events_df['isGoal'] = events_df['isGoal'].replace(np.nan, False).infer_objects()
        else:
            events_df['isGoal'] = False

    # add player name column
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        events_df.loc[events_df.playerId.notna(), 'playerId'] = events_df.loc[events_df.playerId.notna(), 'playerId'].astype(int).astype(str)    
    player_name_col = events_df.loc[:, 'playerId'].map(data['playerIdNameDictionary']) 
    events_df.insert(loc=events_df.columns.get_loc("playerId")+1, column='playerName', value=player_name_col)

    # add home/away column
    h_a_col = events_df['teamId'].map({data['home']['teamId']:'h', data['away']['teamId']:'a'})
    events_df.insert(loc=events_df.columns.get_loc("teamId")+1, column='h_a', value=h_a_col)


    # adding shot body part column
    events_df['shotBodyType'] =  np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        for i in events_df.loc[events_df.isShot==True].index:
            for j in events_df.loc[events_df.isShot==True].qualifiers.loc[i]:
                if j['type'] == 'RightFoot' or j['type'] == 'LeftFoot' or j['type'] == 'Head' or j['type'] == 'OtherBodyPart':
                    events_df.loc[i, 'shotBodyType'] = j['type']


    # adding shot situation column
    events_df['situation'] =  np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        for i in events_df.loc[events_df.isShot==True].index:
            for j in events_df.loc[events_df.isShot==True].qualifiers.loc[i]:
                if j['type'] == 'FromCorner' or j['type'] == 'SetPiece' or j['type'] == 'DirectFreekick':
                    events_df.loc[i, 'situation'] = j['type']
                if j['type'] == 'RegularPlay':
                    events_df.loc[i, 'situation'] = 'OpenPlay' 

    event_types = list(data['matchCentreEventTypeJson'].keys())
    event_type_cols = pd.DataFrame({event_type: pd.Series([event_type in row for row in events_df['satisfiedEventsTypes']]) for event_type in event_types})
    events_df = pd.concat([events_df, event_type_cols], axis=1)


    return events_df

In [42]:
for i in range(20):
    print(matches_df['events'][i])

{'id': 2744721261.0, 'eventId': 2, 'minute': 0, 'second': 0, 'teamId': 52, 'x': 0.0, 'y': 0.0, 'expandedMinute': 0, 'period': {'value': 1, 'displayName': 'FirstHalf'}, 'type': {'value': 32, 'displayName': 'Start'}, 'outcomeType': {'value': 1, 'displayName': 'Successful'}, 'qualifiers': [], 'satisfiedEventsTypes': [], 'isTouch': False, 'matchId': 1866143, 'startDate': '2024-11-05T00:00:00', 'startTime': '2024-11-05T21:00:00', 'score': '1 : 3', 'ftScore': '1 : 3', 'htScore': '1 : 2', 'etScore': '', 'venueName': 'Santiago Bernabéu', 'maxMinute': 96}
{'id': 2744721279.0, 'eventId': 2, 'minute': 0, 'second': 0, 'teamId': 80, 'x': 0.0, 'y': 0.0, 'expandedMinute': 0, 'period': {'value': 1, 'displayName': 'FirstHalf'}, 'type': {'value': 32, 'displayName': 'Start'}, 'outcomeType': {'value': 1, 'displayName': 'Successful'}, 'qualifiers': [], 'satisfiedEventsTypes': [], 'isTouch': False, 'matchId': 1866143, 'startDate': '2024-11-05T00:00:00', 'startTime': '2024-11-05T21:00:00', 'score': '1 : 3', 